# 03 — Conexión Gmail y prueba del agente

Este notebook trabaja directamente con la cuenta Gmail de pruebas configurada.

La primera ejecución abre el navegador para autorizar mediante OAuth. No se utiliza ni se guarda la contraseña.

## 1. Instalar las librerías de Google

Descomenta la línea si todavía no están instaladas.

In [ ]:
# !pip install -q google-api-python-client google-auth-httplib2 google-auth-oauthlib

## 2. Importar el código del proyecto

Añadimos la carpeta raíz para poder importar los módulos de `src`.

In [ ]:
import sys

if ".." not in sys.path:
    sys.path.append("..")

from src import config
from src.agente import analizar_correo
from src.gmail_client import GmailClient
from src.google_oauth import crear_servicio_gmail

print("Cuenta esperada:", config.EXPECTED_GMAIL_ADDRESS)
print("Modo:", config.GMAIL_ACCESS_MODE)

## 3. Autorizar y comprobar Gmail

Antes de ejecutar esta celda debe existir `../credentials.json`.

In [ ]:
servicio, perfil = crear_servicio_gmail()

print("Cuenta autorizada:", perfil["emailAddress"])
print("Mensajes totales:", perfil.get("messagesTotal"))
print("Hilos totales:", perfil.get("threadsTotal"))

## 4. Buscar correos reales

Por defecto se buscan correos no leídos de la bandeja de entrada.

In [ ]:
gmail = GmailClient(servicio)

mensajes = gmail.buscar_correos(limite=5)

print("Correos encontrados:", len(mensajes))
print(mensajes)

## 5. Leer y analizar el primer correo

Esta celda envía el contenido del correo seleccionado al proveedor LLM configurado.

In [ ]:
if not mensajes:
    print("No hay correos para analizar.")

else:
    correo = gmail.leer_correo(mensajes[0]["id"])

    print("De:", correo["remitente"])
    print("Asunto:", correo["asunto"])
    print("Cuerpo:", correo["cuerpo"][:1000])

    resultado, uso = analizar_correo(correo)

    print("\nRESULTADO")
    print(resultado)
    print("\nTokens totales:", uso.total_tokens)

## 6. Acciones sobre Gmail

Durante la primera prueba deben permanecer:

```env
GMAIL_ACCESS_MODE=lectura
ALLOW_CREATE_DRAFTS=false
ALLOW_MODIFY_LABELS=false
```

Cuando la clasificación esté validada, cambiaremos de forma controlada al modo `gestion` para probar borradores y etiquetas. El proyecto no contiene ninguna función de envío.